# LeetCode 561 — Array Partition bằng Python

**Link đề:** https://leetcode.com/problems/array-partition/
**Độ khó:** Easy · **Chủ đề:** Array, Greedy, Sorting, Counting Sort

```python
class Solution:
    def arrayPairSum(self, nums: List[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu bài toán **ghép cặp** để tối đa tổng các `min` của mỗi cặp.
2. Tự khám phá quy luật bằng **brute force** trên mảng nhỏ, rồi rút ra **tham lam (greedy)**: sắp xếp và ghép hai số liền kề.
3. Nắm `sorted`/`list.sort`, slicing có bước `nums[::2]`, `sum`, `min`, `itertools.permutations`.
4. **Chứng minh** vì sao tham lam đúng (lập luận "đổi chỗ" — exchange argument).
5. Viết thêm **counting sort** `O(n + R)` tận dụng giới hạn giá trị `[-10^4, 10^4]`; so sánh, test đối chiếu.

## 1. Đề bài

Cho mảng `nums` gồm `2n` số nguyên. Chia thành `n` **cặp** `(a1, b1), ..., (an, bn)` sao cho tổng

`min(a1, b1) + min(a2, b2) + ... + min(an, bn)`

là **lớn nhất**. Trả về tổng lớn nhất đó.

| `nums` | Output | Cách ghép tốt nhất |
|---|---:|---|
| `[1,4,3,2]` | 4 | `(1,2), (3,4)` → `1 + 3` |
| `[6,2,6,5,1,2]` | 9 | `(1,2), (2,5), (6,6)` → `1 + 2 + 6` |

Ví dụ 1 liệt kê đủ 3 cách ghép:
- `(1,4),(2,3)` → `1 + 2 = 3`
- `(1,3),(2,4)` → `1 + 2 = 3`
- `(1,2),(3,4)` → `1 + 3 = 4` ← lớn nhất

**Constraints:** `1 <= n <= 10^4`, `nums.length == 2n`, `-10^4 <= nums[i] <= 10^4`.

## 2. Nền tảng Python cần dùng

### 2.1. `min`, `sum`

`min(a, b)` trả số nhỏ hơn; `sum(iterable)` cộng mọi phần tử.

In [1]:
print(min(1, 4), min(6, 6), min(-3, 2))
print(sum([1, 3]), sum([]))

1 6 -3
4 0


### 2.2. Sắp xếp: `sorted(nums)` và `nums.sort()`

- `sorted(nums)` trả về **list mới** đã sắp, không đổi `nums`.
- `nums.sort()` sắp **tại chỗ**, trả về `None`.

Cả hai dùng thuật toán Timsort, `O(n log n)`.

In [2]:
nums = [6, 2, 6, 5, 1, 2]
print(sorted(nums), nums)       # nums giữ nguyên
r = nums.sort()
print(r, nums)                  # sort() trả None, nums đã đổi

[1, 2, 2, 5, 6, 6] [6, 2, 6, 5, 1, 2]
None [1, 2, 2, 5, 6, 6]


### 2.3. Slicing có bước: `a[::2]`

`a[start:stop:step]`. `a[::2]` lấy các phần tử ở vị trí **chẵn** `0, 2, 4, ...`; `a[1::2]` lấy vị trí lẻ.

In [3]:
a = [1, 2, 2, 5, 6, 6]
print(a[::2])    # [1, 2, 6]
print(a[1::2])   # [2, 5, 6]

[1, 2, 6]
[2, 5, 6]


## 3. Cách 1 — Brute force: thử mọi cách ghép (để khám phá quy luật)

Cách nghĩ đầu tiên: liệt kê **mọi** cách chia thành cặp, tính tổng, lấy max.

Cách liệt kê đệ quy: lấy phần tử đầu tiên, ghép với **từng** phần tử còn lại; phần còn lại giải tiếp bằng đệ quy.

Số cách ghép `2n` phần tử = `(2n-1) × (2n-3) × ... × 1` — tăng cực nhanh (n=10 đã ~6.5 triệu). Chỉ dùng để **thử nghiệm** trên mảng nhỏ.

In [4]:
def best_pairing_brute(nums: list[int]) -> int:
    if not nums:
        return 0
    first, rest = nums[0], nums[1:]
    best = float("-inf")
    for j in range(len(rest)):
        partner = rest[j]
        remaining = rest[:j] + rest[j + 1:]
        best = max(best, min(first, partner) + best_pairing_brute(remaining))
    return best

print(best_pairing_brute([1, 4, 3, 2]))         # 4
print(best_pairing_brute([6, 2, 6, 5, 1, 2]))   # 9

4
9


`float("-inf")` là "âm vô cùng" — nhỏ hơn mọi số, dùng làm giá trị khởi tạo khi tìm max. `rest[:j] + rest[j+1:]` tạo list mới bỏ phần tử ở vị trí `j`.

Thử in ra **cách ghép tốt nhất** cho vài mảng đã sắp xếp để tìm quy luật:

In [5]:
def best_pairing_with_pairs(nums):
    if not nums:
        return 0, []
    first, rest = nums[0], nums[1:]
    best = (float("-inf"), [])
    for j in range(len(rest)):
        sub_sum, sub_pairs = best_pairing_with_pairs(rest[:j] + rest[j + 1:])
        total = min(first, rest[j]) + sub_sum
        if total > best[0]:
            best = (total, [(first, rest[j])] + sub_pairs)
    return best

for arr in [[1, 2, 3, 4], [1, 2, 2, 5, 6, 6], [1, 3, 4, 8, 9, 10], [-5, -1, 0, 7]]:
    print(arr, "->", best_pairing_with_pairs(arr))

[1, 2, 3, 4] -> (4, [(1, 2), (3, 4)])
[1, 2, 2, 5, 6, 6] -> (9, [(1, 2), (2, 5), (6, 6)])
[1, 3, 4, 8, 9, 10] -> (14, [(1, 3), (4, 8), (9, 10)])
[-5, -1, 0, 7] -> (-5, [(-5, -1), (0, 7)])


**Quy luật lộ ra:** với mảng **đã sắp xếp**, cách ghép tốt nhất luôn là ghép **hai số liền kề**: `(a0,a1), (a2,a3), ...`. Khi đó `min` của mỗi cặp là số đứng ở vị trí **chẵn**, nên đáp án = `a0 + a2 + a4 + ...`.

## 4. Insight — Vì sao ghép liền kề sau khi sắp xếp là tốt nhất?

Trực giác: số **nhỏ nhất** `a0` chắc chắn bị "tính" (nó là min của cặp chứa nó, dù ghép với ai). Ghép nó với một số lớn là **lãng phí** số lớn đó — số lớn sẽ không bao giờ được tính. Vậy ta nên "hy sinh" số **nhỏ thứ hai** `a1` cho `a0`, giữ các số lớn để chúng làm `min` của cặp khác.

Gợi ý của LeetCode cũng nói: *"Minimum element gets added into the result in sacrifice of maximum element."*

Lặp lại lập luận cho phần còn lại `a2, a3, ...` ⇒ ghép `(a2,a3)`, rồi `(a4,a5)`...

Mục 9 sẽ chứng minh chặt chẽ.

## 5. Cách 2 — Sắp xếp + cộng vị trí chẵn (lời giải chính)

1. Sắp xếp `nums`.
2. Cộng các phần tử ở vị trí `0, 2, 4, ...`.

Hai cách viết bước 2: vòng `for i in range(0, len(nums), 2)` hoặc slicing `nums[::2]`.

In [6]:
def array_pair_sum_sort(nums: list[int]) -> int:
    a = sorted(nums)
    total = 0
    for i in range(0, len(a), 2):
        total += a[i]           # a[i] = min(a[i], a[i+1]) vì đã sắp xếp
    return total

def array_pair_sum_slice(nums: list[int]) -> int:
    return sum(sorted(nums)[::2])

print(array_pair_sum_sort([1, 4, 3, 2]), array_pair_sum_slice([1, 4, 3, 2]))
print(array_pair_sum_sort([6, 2, 6, 5, 1, 2]), array_pair_sum_slice([6, 2, 6, 5, 1, 2]))

4 4
9 9


## 6. Trace

`nums = [6, 2, 6, 5, 1, 2]` → sắp xếp `[1, 2, 2, 5, 6, 6]`.

| i | cặp `(a[i], a[i+1])` | `min` = `a[i]` | total |
|---:|---|---:|---:|
| 0 | (1, 2) | 1 | 1 |
| 2 | (2, 5) | 2 | 3 |
| 4 | (6, 6) | 6 | **9** |

In [7]:
def trace_pair_sum(nums: list[int]) -> int:
    a = sorted(nums)
    print("sau sắp xếp:", a)
    total = 0
    for i in range(0, len(a), 2):
        total += a[i]
        print(f"i={i}: cặp ({a[i]}, {a[i + 1]}) -> min = {a[i]}, total = {total}")
    return total

trace_pair_sum([6, 2, 6, 5, 1, 2])
print()
trace_pair_sum([-5, 7, 0, -1])

sau sắp xếp: [1, 2, 2, 5, 6, 6]
i=0: cặp (1, 2) -> min = 1, total = 1
i=2: cặp (2, 5) -> min = 2, total = 3
i=4: cặp (6, 6) -> min = 6, total = 9

sau sắp xếp: [-5, -1, 0, 7]
i=0: cặp (-5, -1) -> min = -5, total = -5
i=2: cặp (0, 7) -> min = 0, total = -5


-5

## 7. Code nộp LeetCode — Sắp xếp

In [8]:
from typing import List

class Solution:
    def arrayPairSum(self, nums: List[int]) -> int:
        nums.sort()
        return sum(nums[::2])

print(Solution().arrayPairSum([1, 4, 3, 2]))
print(Solution().arrayPairSum([6, 2, 6, 5, 1, 2]))

4
9


`nums.sort()` sửa trực tiếp mảng đầu vào — chấp nhận được trên LeetCode. Nếu không muốn đổi input, dùng `sorted(nums)` (tốn thêm `O(n)` bộ nhớ).

## 8. Cách 3 — Counting sort `O(n + R)`

Giá trị chỉ nằm trong `[-10^4, 10^4]` → có `R = 20001` giá trị khả dĩ. Thay vì sắp xếp bằng so sánh `O(n log n)`, ta **đếm** số lần xuất hiện của mỗi giá trị, rồi duyệt giá trị từ nhỏ đến lớn — giống như đọc mảng đã sắp xếp.

Ta không cần dựng lại mảng: chỉ cần biết phần tử thứ `k` (theo thứ tự tăng) có ở vị trí **chẵn** hay không. Dùng cờ `take` (True/False) đảo sau mỗi phần tử.

- `count[v + 10000]`: số lần giá trị `v` xuất hiện (cộng `10000` để chỉ số không âm — gọi là **offset**).

In [9]:
OFFSET = 10_000
print(-10_000 + OFFSET, 0 + OFFSET, 10_000 + OFFSET)   # chỉ số 0, 10000, 20000

0 10000 20000


`10_000` là cách viết số có dấu gạch dưới cho dễ đọc, bằng `10000`.

In [10]:
def array_pair_sum_counting(nums: list[int]) -> int:
    OFFSET = 10_000
    count = [0] * (2 * OFFSET + 1)
    for x in nums:
        count[x + OFFSET] += 1

    total = 0
    take = True                    # phần tử tiếp theo (theo thứ tự tăng) có ở vị trí chẵn không?
    for idx in range(len(count)):
        c = count[idx]
        while c > 0:
            if take:
                total += idx - OFFSET
            take = not take
            c -= 1
    return total

print(array_pair_sum_counting([1, 4, 3, 2]), array_pair_sum_counting([6, 2, 6, 5, 1, 2]))

4 9


In [11]:
def trace_counting(nums):
    OFFSET = 10_000
    count = {}
    for x in nums:
        count[x] = count.get(x, 0) + 1
    print("bảng đếm (chỉ in giá trị có xuất hiện):", dict(sorted(count.items())))
    total, take, k = 0, True, 0
    for v in sorted(count):
        for _ in range(count[v]):
            print(f"phần tử thứ {k} = {v}: {'LẤY' if take else 'bỏ'}")
            if take:
                total += v
            take = not take
            k += 1
    print("total =", total)
    return total

trace_counting([6, 2, 6, 5, 1, 2])

bảng đếm (chỉ in giá trị có xuất hiện): {1: 1, 2: 2, 5: 1, 6: 2}
phần tử thứ 0 = 1: LẤY
phần tử thứ 1 = 2: bỏ
phần tử thứ 2 = 2: LẤY
phần tử thứ 3 = 5: bỏ
phần tử thứ 4 = 6: LẤY
phần tử thứ 5 = 6: bỏ
total = 9


9

### Tối ưu nhỏ: xử lý cả nhóm trùng một lúc

Thay vì `while c > 0` từng phần tử, ta tính số phần tử được lấy trong nhóm `c` giá trị bằng nhau:
- Nếu đang `take`: lấy `(c + 1) // 2` phần tử; ngược lại lấy `c // 2`.
- Sau nhóm, `take` đảo nếu `c` lẻ.

Thời gian vẫn `O(n + R)` nhưng vòng trong biến mất.

In [12]:
def array_pair_sum_counting_fast(nums: list[int]) -> int:
    OFFSET = 10_000
    count = [0] * (2 * OFFSET + 1)
    for x in nums:
        count[x + OFFSET] += 1
    total, take = 0, True
    for idx, c in enumerate(count):
        if c:
            k = (c + 1) // 2 if take else c // 2
            total += k * (idx - OFFSET)
            if c % 2 == 1:
                take = not take
    return total

print(array_pair_sum_counting_fast([1, 4, 3, 2]), array_pair_sum_counting_fast([6, 2, 6, 5, 1, 2]))

4 9


### Code nộp LeetCode — Counting sort

In [13]:
class Solution:
    def arrayPairSum(self, nums: List[int]) -> int:
        OFFSET = 10_000
        count = [0] * (2 * OFFSET + 1)
        for x in nums:
            count[x + OFFSET] += 1
        total, take = 0, True
        for idx, c in enumerate(count):
            if c:
                k = (c + 1) // 2 if take else c // 2
                total += k * (idx - OFFSET)
                if c % 2:
                    take = not take
        return total

print(Solution().arrayPairSum([6, 2, 6, 5, 1, 2]))

9


## 9. Vì sao tham lam đúng? (exchange argument)

Gọi mảng đã sắp `a0 ≤ a1 ≤ ... ≤ a(2n-1)`.

**Bổ đề:** tồn tại một cách ghép tối ưu trong đó `a0` ghép với `a1`.

Chứng minh: lấy một cách ghép tối ưu bất kỳ. Nếu `a0` đã ghép với `a1` thì xong. Nếu không, `a0` ghép với `x`, còn `a1` ghép với `y` (`x, y ≥ a1 ≥ a0`). Đổi thành `(a0, a1)` và `(x, y)`:
- Trước: `min(a0, x) + min(a1, y) = a0 + a1`.
- Sau: `min(a0, a1) + min(x, y) = a0 + min(x, y) ≥ a0 + a1` (vì `x, y ≥ a1`).

Tổng **không giảm** ⇒ cách ghép mới cũng tối ưu và có `(a0, a1)`. ∎

Bỏ cặp `(a0, a1)` đi, phần còn lại là bài toán cùng dạng trên `a2..a(2n-1)` → áp dụng lại bổ đề (quy nạp). Kết quả: ghép `(a0,a1), (a2,a3), ...` là tối ưu, tổng = `a0 + a2 + ... + a(2n-2)`.

**Một cách nhìn khác:** tổng các `min` = `(tổng mọi số − tổng các chênh lệch |a−b| của từng cặp) / 2`. Muốn tổng min lớn nhất ⇔ tổng chênh lệch nhỏ nhất ⇔ ghép các số **gần nhau** nhất ⇔ ghép liền kề sau khi sắp xếp.

In [14]:
# Kiểm tra công thức min(a, b) = (a + b - |a - b|) / 2
for a_, b_ in [(1, 4), (6, 6), (-3, 2)]:
    print(a_, b_, min(a_, b_), (a_ + b_ - abs(a_ - b_)) // 2)

1 4 1 1
6 6 6 6
-3 2 -3 -3


## 10. So sánh và độ phức tạp

Gọi `m = len(nums) = 2n`, `R = 20001` là số giá trị khả dĩ.

| Cách | Time | Space thêm | Ghi chú |
|---|---|---|---|
| 1. Brute force mọi cách ghép | `O((m-1)!!)` (giai thừa kép) | `O(m)` | chỉ để khám phá |
| 2. Sắp xếp + `[::2]` | `O(m log m)` | `O(1)` nếu `sort()` tại chỗ (Timsort cần tới `O(m)` phụ) | **ngắn, khuyên dùng** |
| 3. Counting sort | `O(m + R)` | `O(R)` | nhanh khi `R` nhỏ |

Với `m ≤ 2·10^4`, cả cách 2 và 3 đều rất nhanh. Cách 3 minh họa ý tưởng: **giới hạn giá trị nhỏ ⇒ đếm thay vì so sánh**.

## 11. Bộ test: mẫu đề, biên, đối chiếu brute force

In [15]:
tests = [
    ([1, 4, 3, 2], 4),
    ([6, 2, 6, 5, 1, 2], 9),
    ([1, 2], 1),
    ([5, 5], 5),
    ([-1, -2], -2),
    ([-10000, 10000], -10000),
    ([7, 7, 7, 7], 14),
    ([-5, 7, 0, -1], -5 + 0),
]
methods = [best_pairing_brute, array_pair_sum_sort, array_pair_sum_slice,
           array_pair_sum_counting, array_pair_sum_counting_fast]
for nums, expected in tests:
    for f in methods:
        assert f(list(nums)) == expected, (f.__name__, nums)
    print(f"{str(nums):<22} -> {expected}")
print("Test mẫu PASS!")

[1, 4, 3, 2]           -> 4
[6, 2, 6, 5, 1, 2]     -> 9
[1, 2]                 -> 1
[5, 5]                 -> 5
[-1, -2]               -> -2
[-10000, 10000]        -> -10000
[7, 7, 7, 7]           -> 14
[-5, 7, 0, -1]         -> -5
Test mẫu PASS!


In [16]:
import random

random.seed(561)
for _ in range(400):
    m = 2 * random.randint(1, 4)                  # brute force chỉ chạy được với mảng nhỏ
    nums = [random.randint(-6, 6) for _ in range(m)]
    expected = best_pairing_brute(nums)
    for f in methods[1:]:
        assert f(list(nums)) == expected, (f.__name__, nums)
for _ in range(100):
    nums = [random.randint(-10000, 10000) for _ in range(2 * random.randint(1, 500))]
    assert array_pair_sum_slice(nums) == array_pair_sum_counting(nums) == array_pair_sum_counting_fast(nums)
print("Đối chiếu brute force và mảng lớn: PASS")

Đối chiếu brute force và mảng lớn: PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Cộng vị trí lẻ (lấy max của mỗi cặp)

In [17]:
nums = [1, 4, 3, 2]
print("sai :", sum(sorted(nums)[1::2]), "| đúng:", sum(sorted(nums)[::2]))

sai : 6 | đúng: 4


Vì sao sai: sau sắp xếp, phần tử vị trí lẻ là **max** của cặp. Đề cần **min**.

### Lỗi 2: Ghép số nhỏ nhất với số lớn nhất ("cân bằng")

Nhiều người nghĩ ghép nhỏ–lớn cho "công bằng". Thực tế đó lại là cách **tệ**.

In [18]:
def wrong_small_big(nums):
    a = sorted(nums)
    n = len(a)
    return sum(min(a[i], a[n - 1 - i]) for i in range(n // 2))

for nums in [[1, 4, 3, 2], [6, 2, 6, 5, 1, 2]]:
    print(nums, "sai:", wrong_small_big(nums), "| đúng:", array_pair_sum_slice(nums))

[1, 4, 3, 2] sai: 3 | đúng: 4
[6, 2, 6, 5, 1, 2] sai: 5 | đúng: 9


### Lỗi 3: Quên sắp xếp

`sum(nums[::2])` trên mảng **chưa** sắp → kết quả phụ thuộc thứ tự input, sai.

In [19]:
nums = [4, 1, 3, 2]
print("quên sort:", sum(nums[::2]), "| đúng:", sum(sorted(nums)[::2]))

quên sort: 7 | đúng: 4


### Lỗi 4: Viết `nums = nums.sort()`

`sort()` trả về `None` → `nums` thành `None`, dòng sau lỗi `TypeError`.

In [20]:
nums = [3, 1, 2, 4]
nums = nums.sort()
print(nums)
try:
    print(nums[::2])
except TypeError as e:
    print("TypeError:", e)

None
TypeError: 'NoneType' object is not subscriptable


### Lỗi 5: Counting sort quên offset cho số âm

`count[x]` với `x = -3` sẽ truy cập `count[-3]` — **chỉ số âm trong Python đếm từ cuối**, không báo lỗi mà âm thầm sai! Luôn cộng `OFFSET`.

## 13. Mẹo nhận dạng pattern

- Bài "**ghép cặp** để tối đa/tối thiểu tổng `min`/`max`/chênh lệch" → thử **sắp xếp** rồi ghép liền kề hoặc ghép hai đầu.
- Không chắc tham lam đúng → viết **brute force** cho mảng nhỏ, in cách ghép tối ưu, tìm quy luật; rồi chứng minh bằng **exchange argument**.
- Giá trị bị chặn trong khoảng nhỏ → **counting sort** `O(n + R)`.

## 14. Biến thể

1. **LeetCode 1877 – Minimize Maximum Pair Sum in Array**: ghép **nhỏ nhất với lớn nhất** (ngược lại bài này!).
2. **LeetCode 881 – Boats to Save People**: hai con trỏ hai đầu sau sắp xếp.
3. **LeetCode 455 – Assign Cookies**: tham lam sau sắp xếp.
4. **LeetCode 2144 – Minimum Cost of Buying Candies With Discount**: sắp giảm dần, bỏ mỗi phần tử thứ 3.

In [21]:
# Biến thể 1 demo: tối thiểu hóa tổng cặp lớn nhất -> ghép nhỏ nhất với lớn nhất
def min_max_pair_sum(nums):
    a = sorted(nums)
    n = len(a)
    return max(a[i] + a[n - 1 - i] for i in range(n // 2))

print(min_max_pair_sum([3, 5, 2, 3]))      # 7
print(min_max_pair_sum([3, 5, 4, 2, 4, 6]))  # 8

7
8


Hai bài trông giống nhau nhưng chiến lược ghép **ngược nhau** — vì mục tiêu khác nhau (tối đa tổng `min` vs tối thiểu `max` của tổng cặp). Đừng học thuộc, hãy chứng minh!

## 15. Bài tập tự luyện

### Bài 1 (dễ)
Viết `sum_even_positions(a)` cộng các phần tử ở vị trí chẵn **không dùng slicing**.

In [22]:
def sum_even_positions(a: list[int]) -> int:
    # TODO
    pass

# assert sum_even_positions([1, 2, 2, 5, 6, 6]) == 9
# assert sum_even_positions([]) == 0

### Bài 2 (dễ)
Viết `optimal_pairs(nums)` trả về **list các cặp** của cách ghép tối ưu (mỗi cặp là tuple `(nhỏ, lớn)`, theo thứ tự tăng).

In [23]:
def optimal_pairs(nums: list[int]) -> list[tuple]:
    # TODO
    pass

# assert optimal_pairs([6, 2, 6, 5, 1, 2]) == [(1, 2), (2, 5), (6, 6)]

### Bài 3 (vừa)
Viết `array_pair_sum_max(nums)`: ghép cặp sao cho tổng các **max** của mỗi cặp là **nhỏ nhất**. Tìm quy luật và chứng minh ngắn gọn.

In [24]:
def array_pair_sum_max(nums: list[int]) -> int:
    # TODO
    pass

# assert array_pair_sum_max([1, 4, 3, 2]) == 6       # (1,2),(3,4) -> 2 + 4
# assert array_pair_sum_max([6, 2, 6, 5, 1, 2]) == 13 # 2 + 5 + 6

### Bài 4 (vừa)
Mở rộng: chia `3n` số thành `n` **bộ ba**, tối đa tổng `min` của mỗi bộ. Viết `triple_min_sum(nums)`.

In [25]:
def triple_min_sum(nums: list[int]) -> int:
    # TODO
    pass

# assert triple_min_sum([1, 2, 3, 4, 5, 6]) == 1 + 4
# assert triple_min_sum([9, 1, 1, 9, 9, 1]) == 1 + 9

### Bài 5 (khó hơn)
Viết `min_total_gap(nums)`: tổng nhỏ nhất của `|a - b|` trên mọi cặp khi chia `2n` số thành `n` cặp. Rồi kiểm tra công thức `arrayPairSum = (sum(nums) - min_total_gap(nums)) // 2`.

In [26]:
def min_total_gap(nums: list[int]) -> int:
    # TODO
    pass

# assert min_total_gap([1, 4, 3, 2]) == 2
# assert (sum([6, 2, 6, 5, 1, 2]) - min_total_gap([6, 2, 6, 5, 1, 2])) // 2 == 9

## 16. Gợi ý 3 mức

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `range(0, len(a), 2)` | Biến `total = 0` | Cộng `a[i]` |
| 2 | Sắp xếp trước | Duyệt bước 2 | `(a[i], a[i+1])` |
| 3 | Số **lớn nhất** chắc chắn bị tính | Hy sinh số lớn thứ hai cho nó | Vẫn ghép liền kề, cộng vị trí **lẻ** |
| 4 | Mỗi bộ ba "lãng phí" hai số | Số nhỏ nhất đi với hai số nhỏ tiếp theo | Sắp xếp, cộng `a[0], a[3], a[6], ...` |
| 5 | Ghép số gần nhau | Sắp xếp, ghép liền kề | Tổng `a[i+1] - a[i]` với `i` chẵn |

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [27]:
def answer_sum_even_positions(a: list[int]) -> int:
    total = 0
    for i in range(0, len(a), 2):
        total += a[i]
    return total

def answer_optimal_pairs(nums: list[int]) -> list[tuple]:
    a = sorted(nums)
    return [(a[i], a[i + 1]) for i in range(0, len(a), 2)]

def answer_array_pair_sum_max(nums: list[int]) -> int:
    return sum(sorted(nums)[1::2])

def answer_triple_min_sum(nums: list[int]) -> int:
    return sum(sorted(nums)[::3])

def answer_min_total_gap(nums: list[int]) -> int:
    a = sorted(nums)
    return sum(a[i + 1] - a[i] for i in range(0, len(a), 2))

assert answer_sum_even_positions([1, 2, 2, 5, 6, 6]) == 9
assert answer_sum_even_positions([]) == 0
assert answer_optimal_pairs([6, 2, 6, 5, 1, 2]) == [(1, 2), (2, 5), (6, 6)]
assert answer_array_pair_sum_max([1, 4, 3, 2]) == 6
assert answer_array_pair_sum_max([6, 2, 6, 5, 1, 2]) == 13
assert answer_triple_min_sum([1, 2, 3, 4, 5, 6]) == 5
assert answer_triple_min_sum([9, 1, 1, 9, 9, 1]) == 10
assert answer_min_total_gap([1, 4, 3, 2]) == 2
for _ in range(300):
    nums = [random.randint(-50, 50) for _ in range(2 * random.randint(1, 20))]
    assert (sum(nums) - answer_min_total_gap(nums)) // 2 == array_pair_sum_slice(nums)
print("Đáp án chạy PASS toàn bộ test!")

Đáp án chạy PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist

```python
class Solution:
    def arrayPairSum(self, nums: List[int]) -> int:
        nums.sort()
        return sum(nums[::2])
```

### Checklist trước khi submit
- [ ] Đã **sắp xếp** trước khi lấy vị trí chẵn.
- [ ] Lấy vị trí **chẵn** `[::2]` (min), không phải lẻ.
- [ ] Không viết `nums = nums.sort()`.
- [ ] Số âm: sắp xếp vẫn đúng; counting sort phải có `OFFSET`.
- [ ] Hiểu được **vì sao** tham lam đúng (đổi chỗ không làm tổng giảm).